In [12]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — ICICI Bank Daily Data + RSI (From 2010)
# Ticker: ICICIBANK.NS (NSE)
# ============================================

In [13]:
df = yf.download("ICICIBANK.NS", 
                 start="2010-01-01", 
                 progress=False,
                 auto_adjust=False,      # helps sometimes
                 repair=True)           # new repair option
df.columns=df.columns.get_level_values(0)

In [14]:
data=pd.DataFrame()
data["Open"] =df["Open"]
data["High"] =df["High"]
data["Low"] =df["Low"]
data["Close"] =df["Close"]
data["Volume"] =df["Volume"]

# --- Calculate RSI (14-period) ---


In [15]:
period = 14
delta = data["Close"].diff()

In [16]:
gain=delta.where(delta>0,0)
loss = -delta.where(delta<0,0)

In [17]:
avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()

In [18]:
rs=avg_gain/avg_loss
data["RSI_14"]=100-(100/(1+rs))

In [19]:
data=data.dropna()
latest=data.iloc[-1]

In [20]:
print(f"✅ ICICI Bank: {len(data)} trading days ({data.index[0].date()} to {data.index[-1].date()})")
print(f"   Latest Close: ₹{float(latest['Close']):.2f}")
print(f"   Latest RSI: {float(latest['RSI_14']):.2f}")
print(f"   Latest Volume: {int(latest['Volume']):,}")
 


✅ ICICI Bank: 4039 trading days (2010-01-21 to 2026-06-02)
   Latest Close: ₹1226.60
   Latest RSI: 38.19
   Latest Volume: 17,920,659


In [21]:
data.tail()

,Open,High,Low,Close,Volume,RSI_14
Date,,,,,,
2026-05-27,1286.000000,1293.400024,1266.699951,1272.699951,18366272,50.216119
2026-05-28,1272.699951,1272.699951,1272.699951,1272.699951,0,50.216119
2026-05-29,1275.099976,1287.800049,1247.900024,1256.400024,31007884,45.495534
2026-06-01,1257.099976,1261.900024,1235.900024,1239.699951,9911975,41.220109
2026-06-02,1233.000000,1237.699951,1220.699951,1226.599976,17920659,38.188457


In [22]:
data.to_csv("icici_bank_daily.csv")

1h Time frame

In [23]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — ICICI Bank Intraday Data + RSI
# 1H: Max ~730 days back (yfinance limit)
# 4H: Aggregated from 1H data
# ============================================

TICKER = "ICICIBANK.NS"

def calculate_rsi(close, period=14):
    delta = close.diff()
    gain = delta.where(delta > 0, 0)
    loss = -delta.where(delta < 0, 0)
    avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
    rs = avg_gain / avg_loss
    return 100 - (100 / (1 + rs))

# -----------------------------------------------
# PART 1: Pull 1H data (max available)
# -----------------------------------------------
print("Pulling 1H data...")
df_1h = yf.download(TICKER, period="max", interval="1h", progress=False)
df_1h.columns = df_1h.columns.get_level_values(0)

data_1h = pd.DataFrame()
data_1h["Open"] = df_1h["Open"]
data_1h["High"] = df_1h["High"]
data_1h["Low"] = df_1h["Low"]
data_1h["Close"] = df_1h["Close"]
data_1h["Volume"] = df_1h["Volume"]
data_1h["RSI_14"] = calculate_rsi(data_1h["Close"])
data_1h = data_1h.dropna()

print(f"✅ 1H: {len(data_1h)} candles ({data_1h.index[0]} to {data_1h.index[-1]})")
print(f"   Latest Close: ₹{float(data_1h['Close'].iloc[-1]):.2f}")
print(f"   Latest RSI: {float(data_1h['RSI_14'].iloc[-1]):.2f}")

data_1h.to_csv("icici_1h.csv")
print(f"   💾 Saved: icici_1h.csv\n")

# -----------------------------------------------
# PART 2: Aggregate to 4H from 1H data
# -----------------------------------------------
print("Aggregating to 4H...")

data_4h = data_1h.resample("4h").agg({
    "Open": "first",
    "High": "max",
    "Low": "min",
    "Close": "last",
    "Volume": "sum"
}).dropna()

# Recalculate RSI on 4H candles
data_4h["RSI_14"] = calculate_rsi(data_4h["Close"])
data_4h = data_4h.dropna()

# Remove non-market hours (rows where Open/Close are same or volume is 0)
data_4h = data_4h[data_4h["Volume"] > 0]

print(f"✅ 4H: {len(data_4h)} candles ({data_4h.index[0]} to {data_4h.index[-1]})")
print(f"   Latest Close: ₹{float(data_4h['Close'].iloc[-1]):.2f}")
print(f"   Latest RSI: {float(data_4h['RSI_14'].iloc[-1]):.2f}")

data_4h.to_csv("icici_4h.csv")
print(f"   💾 Saved: icici_4h.csv\n")

# -----------------------------------------------
# Summary
# -----------------------------------------------
print("=" * 60)
print("Summary:")
print(f"  1H candles: {len(data_1h)} (from {data_1h.index[0].date()})")
print(f"  4H candles: {len(data_4h)} (from {data_4h.index[0].date()})")
print("=" * 60)
print("\nNOTE: yfinance only provides ~730 days of 1H data.")
print("For older intraday data, options are:")
print("  1. Zerodha Kite API (historical data endpoint)")
print("  2. TrueData / GlobalDataFeeds (paid, Indian market specialist)")
print("  3. Store data daily going forward to build your own database")

Pulling 1H data...
✅ 1H: 3415 candles (2024-06-04 09:45:00+00:00 to 2026-06-02 09:45:00+00:00)
   Latest Close: ₹1230.10
   Latest RSI: 35.33
   💾 Saved: icici_1h.csv

Aggregating to 4H...
✅ 4H: 1039 candles (2024-06-11 04:00:00+00:00 to 2026-06-02 08:00:00+00:00)
   Latest Close: ₹1230.10
   Latest RSI: 37.18
   💾 Saved: icici_4h.csv

Summary:
  1H candles: 3415 (from 2024-06-04)
  4H candles: 1039 (from 2024-06-11)

NOTE: yfinance only provides ~730 days of 1H data.
For older intraday data, options are:
  1. Zerodha Kite API (historical data endpoint)
  2. TrueData / GlobalDataFeeds (paid, Indian market specialist)
  3. Store data daily going forward to build your own database


## Bannk Nifty dta

In [25]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — Bank Nifty Daily Data + RSI (From 2010)
# Ticker: ^NSEBANK (NSE Bank Nifty Index)
# ============================================

df = yf.download("^NSEBANK", start="2010-01-01", progress=False, auto_adjust=False, repair=True)
df.columns = df.columns.get_level_values(0)

data = pd.DataFrame()
data["Open"] = df["Open"]
data["High"] = df["High"]
data["Low"] = df["Low"]
data["Close"] = df["Close"]
data["Volume"] = df["Volume"]

# --- RSI with Wilder Smoothing (matches TradingView) ---
period = 14
delta = data["Close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
rs = avg_gain / avg_loss
data["RSI_14"] = 100 - (100 / (1 + rs))

data = data.dropna()
latest = data.iloc[-1]

print(f"✅ Bank Nifty: {len(data)} trading days ({data.index[0].date()} to {data.index[-1].date()})")
print(f"   Latest Close: {float(latest['Close']):.2f}")
print(f"   Latest RSI: {float(latest['RSI_14']):.2f}")
print(f"   Latest Volume: {int(latest['Volume']):,}")

data.to_csv("bank_nifty_daily.csv")
print(f"\n💾 Saved: bank_nifty_daily.csv")
print(f"\nLast 10 days:")
print(data.tail(10).round(2).to_string())

✅ Bank Nifty: 4032 trading days (2010-01-21 to 2026-06-02)
   Latest Close: 53714.65
   Latest RSI: 43.62
   Latest Volume: 0

💾 Saved: bank_nifty_daily.csv

Last 10 days:
                Open      High       Low     Close   Volume  RSI_14
Date                                                               
2026-05-19  53553.75  53770.65  53337.05  53409.15   267800   39.99
2026-05-20  53015.70  53640.90  52836.10  53562.20   221400   41.29
2026-05-21  53963.10  54109.15  53156.15  53439.40   285500   40.53
2026-05-22  53483.85  54213.05  53483.55  54055.35   229300   45.87
2026-05-25  54610.55  55405.20  54590.70  55293.65   262400   54.68
2026-05-26  55311.80  55536.80  54979.75  55092.90   321300   53.17
2026-05-27  54992.95  55221.70  54738.60  54853.85   268100   51.35
2026-05-29  54748.30  55184.45  54116.15  54239.20  1042600   46.90
2026-06-01  54403.85  54582.75  53470.00  53643.10   322900   43.02
2026-06-02  53265.10  53933.55  53121.85  53714.65        0   43.62


## NIFTY Daily Data

In [1]:
import yfinance as yf
import pandas as pd

# ============================================
# QOIN — Nifty 50 Daily Data + RSI (From 2010)
# Ticker: ^NSEI (NSE Nifty 50 Index)
# ============================================

df = yf.download("^NSEI", start="2010-01-01", progress=False,auto_adjust=False)
df.columns = df.columns.get_level_values(0)

data = pd.DataFrame()
data["Open"] = df["Open"]
data["High"] = df["High"]
data["Low"] = df["Low"]
data["Close"] = df["Close"]
data["Volume"] = df["Volume"]

# --- RSI with Wilder Smoothing (14-period, matches TradingView) ---
period = 14
delta = data["Close"].diff()
gain = delta.where(delta > 0, 0)
loss = -delta.where(delta < 0, 0)
avg_gain = gain.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
avg_loss = loss.ewm(alpha=1/period, min_periods=period, adjust=False).mean()
rs = avg_gain / avg_loss
data["RSI_14"] = 100 - (100 / (1 + rs))

data = data.dropna()
latest = data.iloc[-1]

print(f"✅ Nifty 50: {len(data)} trading days ({data.index[0].date()} to {data.index[-1].date()})")
print(f"   Latest Close: {float(latest['Close']):.2f}")
print(f"   Latest RSI: {float(latest['RSI_14']):.2f}")
print(f"   Latest Volume: {int(latest['Volume']):,}")

data.to_csv("nifty50_daily.csv")
print(f"\n💾 Saved: nifty50_daily.csv")
print(f"\nLast 10 days:")
print(data.tail(10).round(2).to_string())

✅ Nifty 50: 4016 trading days (2010-01-21 to 2026-06-02)
   Latest Close: 23483.55
   Latest RSI: 42.94
   Latest Volume: 0

💾 Saved: nifty50_daily.csv

Last 10 days:
                Open      High       Low     Close   Volume  RSI_14
Date                                                               
2026-05-19  23675.30  23782.30  23587.20  23618.00   442000   44.69
2026-05-20  23457.25  23690.90  23397.30  23659.00   344100   45.63
2026-05-21  23830.05  23859.90  23596.60  23654.70   348200   45.54
2026-05-22  23671.20  23835.65  23671.00  23719.30   336100   47.18
2026-05-25  23940.25  24054.45  23922.85  24031.70   351200   54.35
2026-05-26  24004.10  24089.80  23885.45  23913.70   387900   51.51
2026-05-27  23880.35  23983.20  23858.25  23907.15   531600   51.35
2026-05-29  23902.15  24002.80  23484.75  23547.75  1198000   43.36
2026-06-01  23654.50  23733.70  23357.95  23382.60   421700   40.26
2026-06-02  23229.15  23556.95  23229.15  23483.55        0   42.94
